# T-332 / T-333 — Reranker fine-tune (SciReviewGen → MiniLM cross-encoder)

**ResearchGPT**, Phase 3C. The **second** trained model, alongside the verifier.

| | |
|---|---|
| Task | T-332 (train) / T-333 (export) |
| Base | `cross-encoder/ms-marco-MiniLM-L-6-v2` (~22M) — the model already in the pipeline |
| Data | SciReviewGen triples with **BM25 hard negatives** (T-330/T-331) |
| Objective | Binary cross-entropy on `(query, abstract) → relevant / not` |
| Runtime | **GPU T4 x2**, ~20–30 min |
| Internet | **must be ON** |

### Why this model earns a slot
`SESSION_CONTEXT §5.6`: Ammann et al. warn their gains *"rely on a strong,
domain-aligned cross-encoder reranker"* — and ours is a 22M MiniLM trained on
**web search**, not science. That is an unanswered weakness sitting in your own
literature review. Fine-tuning converts it into a contribution.

It is also **zero new inference cost** (a weight swap at T-210) and **off the
M1 critical path** — if this run fails, nothing downstream breaks.

### Read before running
* **Upload the triples first.** Unlike the verifier notebook, this one cannot
  rebuild its own data — the BM25 hard-negative mining needs the full corpus.
  See the README: attach `data/rerank/train_triples.jsonl` as a Kaggle Dataset.
* **Hard negatives are the whole point.** If T-334 later shows ~0 improvement,
  suspect the negatives (T-331) before the training config.
* **The adoption gate is out-of-domain** (T-335): this model ships only if it
  beats stock on *your own 25-paper bibliography*, not on SciReviewGen.

## 1 · Configuration

Values are `TRAINING_SPEC §2.3` verbatim. `SEED` must match
`researchgpt.rerank.triples.SEED`, since the triples were built with it.

In [ ]:
BASE_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # ~22M, already in the pipeline

SEED         = 20260827      # must equal researchgpt.rerank.triples.SEED
EPOCHS       = 2             # TRAINING_SPEC 2.3
LEARNING_RATE = 2e-5
BATCH_SIZE   = 32
MAX_LENGTH   = 512           # truncate the ABSTRACT, never the query
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
DEV_FRAC     = 0.10          # held out BY SURVEY, not by row

OUTPUT_DIR   = "/kaggle/working/reranker-scireviewgen"

# Where the uploaded triples land. Kaggle mounts datasets read-only under
# /kaggle/input/<dataset-slug>/. The glob below finds the file wherever it sits.
TRIPLES_GLOB = "/kaggle/input/**/train_triples.jsonl"

print("base model:", BASE_MODEL)

## 2 · Dependencies

`sentence-transformers` supplies `CrossEncoder`, which implements this
objective directly — `TRAINING_SPEC §2.3` notes it needs no custom loop.

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, mod=None):
    try:
        importlib.import_module(mod or pkg.replace("-", "_"))
        print(f"  present : {pkg}")
    except ImportError:
        print(f"  install : {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, mod in [("transformers", "transformers"),
                 ("accelerate>=0.26", "accelerate"),
                 ("sentence-transformers", "sentence_transformers"),
                 ("scikit-learn", "sklearn")]:
    ensure(pkg, mod)

import torch
print("\ntorch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3 · Load the triples and split **by survey**

⚠️ **The split is by `survey_title`, never by row — this is the single thing
most likely to silently invalidate the result.** One survey contributes many
chapter rows, and a row-level split puts chapters of the *same survey* on both
sides, inflating the dev score for free. Measured in Phase 3B: 500 rows spanned
only **63 distinct surveys**.

The query is `survey_title + " " + chapter_heading`, so the survey title is
recoverable from the query itself and the grouping needs no extra field.

In [ ]:
import glob, json, random
from collections import Counter

paths = glob.glob(TRIPLES_GLOB, recursive=True)
assert paths, (
    f"No triples found at {TRIPLES_GLOB}.\n"
    "Attach the Kaggle Dataset containing train_triples.jsonl "
    "(Add Input -> Datasets). See notebook-kaggle-traning/README.md."
)
print("triples file:", paths[0])

rows = [json.loads(line) for line in open(paths[0], encoding="utf-8") if line.strip()]
print(f"{len(rows):,} pairs loaded")

# Group by query, then split those groups by survey title.
by_query = {}
for r in rows:
    by_query.setdefault(r["query"], []).append(r)

def survey_of(query):
    # The query is "<survey title> <chapter heading>"; chapters of one survey
    # share the title prefix. Grouping on a generous prefix keeps them together
    # without needing the raw field.
    return query[:80]

surveys = sorted({survey_of(q) for q in by_query})
rng = random.Random(SEED)
rng.shuffle(surveys)
n_dev = max(1, int(round(len(surveys) * DEV_FRAC)))
dev_surveys = set(surveys[:n_dev])

train_rows, dev_rows = [], []
for query, group in by_query.items():
    (dev_rows if survey_of(query) in dev_surveys else train_rows).extend(group)

assert not ({survey_of(r["query"]) for r in train_rows} &
            {survey_of(r["query"]) for r in dev_rows}), "survey leaked across the split"

for name, rs in [("train", train_rows), ("dev", dev_rows)]:
    c = Counter(r["label"] for r in rs)
    print(f"  {name:6s} {len(rs):7,d} pairs   pos {c[1]:6,d}  neg {c[0]:7,d}  "
          f"queries {len({r['query'] for r in rs}):5,d}")

print(f"\n  OK: {len(surveys)} survey groups, {n_dev} held out, none on both sides")

## 4 · Build the CrossEncoder

`num_labels=1` with `BCEWithLogitsLoss` is the binary relevance objective from
`TRAINING_SPEC §2.3`. This is the same architecture the pipeline already runs at
T-210, so the tuned result is a **weight swap** — nothing downstream changes.

In [ ]:
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CrossEncoderClassificationEvaluator
import torch, numpy as np, random as _random

torch.manual_seed(SEED); np.random.seed(SEED); _random.seed(SEED)

model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=MAX_LENGTH)
print("loaded:", BASE_MODEL)
print("params:", sum(p.numel() for p in model.model.parameters()) / 1e6, "M")

## 5 · Stock baseline on the dev split — **before** training

Taken first, through the identical code path, so the comparison in section 7 is
against the same data rather than a remembered number.

**nDCG@10 is the metric to watch**, not accuracy: this is a ranking task, and a
model can score well on per-pair classification while ordering badly.

In [ ]:
import numpy as np

def group_by_query(rs):
    g = {}
    for r in rs:
        g.setdefault(r["query"], []).append(r)
    return g

def ndcg_at_k(labels_in_rank_order, k=10):
    gains = labels_in_rank_order[:k]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sorted(labels_in_rank_order, reverse=True)[:k]
    idcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_ranking(m, rs, name, k=10):
    groups = group_by_query(rs)
    ndcgs, recalls, precisions = [], [], []
    for query, items in groups.items():
        scores = m.predict([(query, it["text"]) for it in items],
                           batch_size=128, show_progress_bar=False)
        order = np.argsort(-np.asarray(scores))
        ranked = [items[i]["label"] for i in order]
        n_pos = sum(ranked)
        if n_pos == 0:
            continue
        ndcgs.append(ndcg_at_k(ranked, k))
        recalls.append(sum(ranked[:20]) / n_pos)
        precisions.append(sum(ranked[:12]) / 12)
    out = {"nDCG@10": float(np.mean(ndcgs)),
           "Recall@20": float(np.mean(recalls)),
           "Precision@12": float(np.mean(precisions)),
           "queries": len(ndcgs)}
    print(f"  {name:14s} nDCG@10 {out['nDCG@10']:.4f}   "
          f"Recall@20 {out['Recall@20']:.4f}   Precision@12 {out['Precision@12']:.4f}   "
          f"(n={out['queries']})")
    return out

print("ranking metrics on the held-out dev surveys:")
stock_metrics = evaluate_ranking(model, dev_rows, "stock")

## 6 · Train

Binary cross-entropy, 2 epochs, lr 2e-5, batch 32, 10% warmup — `TRAINING_SPEC
§2.3` exactly.

In [ ]:
import math, time
from torch.utils.data import DataLoader
from sentence_transformers import InputExample

train_examples = [InputExample(texts=[r["query"], r["text"]], label=float(r["label"]))
                  for r in train_rows]
train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

steps_per_epoch = math.ceil(len(train_examples) / BATCH_SIZE)
warmup_steps = int(steps_per_epoch * EPOCHS * WARMUP_RATIO)
print(f"{len(train_examples):,} examples | {steps_per_epoch:,} steps/epoch | "
      f"{warmup_steps:,} warmup steps")

t0 = time.time()
model.fit(
    train_dataloader=train_loader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    weight_decay=WEIGHT_DECAY,
    use_amp=torch.cuda.is_available(),
    show_progress_bar=True,
)
train_seconds = time.time() - t0
print(f"\ntrained in {train_seconds/60:.1f} min")

## 7 · In-domain result — stock vs fine-tuned

⚠️ **This is the IN-DOMAIN number and it is not the adoption gate.**
SciReviewGen is frozen at S2ORC 2020-07-05 and skews biomedical; your corpus is
CS/agentic-RAG. **A large gain here with no gain on your own bibliography means
you fitted the benchmark, not the task** (`TRAINING_SPEC §2.4`).

The decision is made locally by T-334/T-335, against your own 25-paper
bibliography, *after* int8 quantization.

In [ ]:
print("ranking metrics on the held-out dev surveys:")
tuned_metrics = evaluate_ranking(model, dev_rows, "fine-tuned")

print("\n" + "=" * 74)
print("T-332 IN-DOMAIN (SciReviewGen dev, survey-disjoint)")
print("=" * 74)
print(f"  {'arm':14s} {'nDCG@10':>10s} {'Recall@20':>11s} {'Precision@12':>13s}")
for name, m in (("stock", stock_metrics), ("fine-tuned", tuned_metrics)):
    print(f"  {name:14s} {m['nDCG@10']:10.4f} {m['Recall@20']:11.4f} "
          f"{m['Precision@12']:13.4f}")
d = {k: tuned_metrics[k] - stock_metrics[k]
     for k in ("nDCG@10", "Recall@20", "Precision@12")}
print(f"  {'delta':14s} {d['nDCG@10']:+10.4f} {d['Recall@20']:+11.4f} "
      f"{d['Precision@12']:+13.4f}")
print("=" * 74)
if d["nDCG@10"] <= 0.001:
    print("~0 in-domain gain. Suspect the HARD NEGATIVES (T-331) first:")
    print("check random_fallback_share_of_negatives in eval/rerank_triples_t330.json.")

## 8 · Save

Download **both** files. The checkpoint feeds T-333 (ONNX export) and T-334/T-335
(the adoption gate); the metrics JSON supplies the in-domain row of the
`TRAINING_SPEC §3` two-model table.

In [ ]:
import os, shutil, json

model.save(OUTPUT_DIR)

metrics = {
    "task": "T-332",
    "base_model": BASE_MODEL,
    "seed": SEED,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "dev_frac": DEV_FRAC,
    "counts": {"train_pairs": len(train_rows), "dev_pairs": len(dev_rows),
               "survey_groups": len(surveys), "dev_surveys": n_dev},
    "in_domain": {"stock": stock_metrics, "fine_tuned": tuned_metrics},
    "train_seconds": train_seconds,
    "note": ("IN-DOMAIN ONLY. The adoption gate (T-335) is out-of-domain, on the "
             "project's own 25-paper bibliography, after int8 quantization."),
}
with open("/kaggle/working/reranker_metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)

shutil.make_archive("/kaggle/working/reranker-scireviewgen", "zip", OUTPUT_DIR)
print("saved:")
for f in sorted(os.listdir("/kaggle/working")):
    p = os.path.join("/kaggle/working", f)
    if os.path.isfile(p):
        print(f"  {f:40s} {os.path.getsize(p)/1e6:8.2f} MB")
print("\nDownload reranker-scireviewgen.zip AND reranker_metrics.json.")
print("Then locally:")
print("  python -m researchgpt.rerank.export_onnx --checkpoint <unzipped dir>")
print("  python -m researchgpt.rerank.evaluate    --tuned      <unzipped dir>")